In [ ]:
import pandas as pd
import sys
import os
from pathlib import Path
import numpy as np
import statsmodels.api as sm
from scipy.stats import spearmanr, t

from scipy.stats import t as t_dist

project_root = Path.cwd().parent 
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    


Vì bắt đầu từ 2020-06-29 mới thực hiện đánh giá, vì mỗi coin cần thời gian 6 tháng để được là eligible

Ở phần này, ta thực hiện từ 2020-09-27 để các factor yêu cầu lấy lịch sử dữ liệu đủ để thưc hiện các kiểm định

In [2]:
df = pd.read_parquet("../data/processed/B1_factor_construction.parquet")


df = df[df["timestamp"] >= "2020-09-27"]
df

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d,fwd_1d,fwd_5d,fwd_14d
4851,2020-09-27 00:00:00+00:00,BTC/USDT,1,0.104937,1.295379,0.608275,-0.643410,0.059951,-0.801625,-1.335920,-1.460214,-1.898988,-1.592089,0.004246,-1.130024,-1.219553,-0.007278,-0.019101,0.053733
4852,2020-09-27 00:00:00+00:00,ETH/USDT,1,-0.304986,0.608550,0.439487,0.332203,0.293393,-0.199482,-0.491809,-0.556091,-0.670204,-1.220259,0.010036,-1.124777,-1.213106,-0.009923,-0.033825,0.045576
4853,2020-09-27 00:00:00+00:00,LINK/USDT,1,2.119062,-0.288078,-0.790309,1.667830,1.710906,1.711488,1.784511,1.734227,1.347208,1.230560,0.045198,-0.919622,-1.009862,-0.048489,-0.157964,0.005322
4854,2020-09-27 00:00:00+00:00,BNB/USDT,1,0.294696,-0.908935,1.626324,0.596841,-0.032517,1.024796,0.163254,0.335636,0.197070,-1.114051,0.001952,-0.950499,-0.984128,0.027596,0.040533,0.083150
4855,2020-09-27 00:00:00+00:00,TRX/USDT,1,0.111595,-0.604139,1.507666,0.486506,-0.995973,-0.528470,-1.229117,-0.538236,0.136846,0.244343,-0.021946,-0.706681,-0.747670,-0.012869,-0.026292,-0.001129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48483,2021-12-31 00:00:00+00:00,ACM/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011087,NaN,NaN,0.036744,-0.060737,-0.031550
48484,2021-12-31 00:00:00+00:00,RIF/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011526,NaN,NaN,0.038076,-0.079707,-0.098440
48485,2021-12-31 00:00:00+00:00,ONG/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011306,NaN,NaN,0.049198,-0.001874,-0.044586
48486,2021-12-31 00:00:00+00:00,ASR/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.007439,NaN,NaN,0.049288,-0.053680,-0.044011


In [2]:
col_forward_return = ["fwd_1d","fwd_5d","fwd_14d"]

col_factor = ["z_momentum_7d", "z_momentum_14d", "z_momentum_30d", "z_momentum_90d", "z_reversal_1d", "z_reversal_3d", "z_vol_7d",
              "z_vol_14d", "z_vol_30d", "z_vol_of_vol_14d", "z_amihud_14d", "z_amihud_30d"]

In [ ]:
df["timestamp"].nunique()


461

# Phần C : Factor Validation

- Mục tiêu lớn nhất của phần C, là đánh giá các chỉ số thống kê của các factor ta đã tạo từ phần B
- Sau khi có các chỉ số thống kê, ta sẽ chỉ lựa chọn các factor có ý nghĩa và mang tính độc lập với nhau

## C1 — Daily Information Coefficient

- Xây dựng ra một dataframe với từng cột lần lượt là timestamp, factor, horizon, coeff_spearman   
- Ứng với ngày thứ t, sẽ là coeff spearman giữa factor và một horizon, mỗi factor trong 1 ngày tương ứng 3 dòng trong dataframe


In [4]:
"""
Mục tiêu của hàm này là tạo ra một dataframe
Tính spearman coeff cho từng foward_return (fwd_1,5,14d)
Ứng với ngày thứ t, với từng factor, ta sẽ tính coeff cho giữa factor và 3 biến foward_return
Với mỗi ngày, mỗi biến sẽ có 3 dòng, vì có 12 factor nên sẽ có 12*3=36 dòng/ ngày và có khoảng 1096*36 dòng 

input là df, col_factor (danh sách các cột factor), col_foward_return (danh sách các cột foward) 
"""

def build_daily_ic(df, col_factor, col_forward_return):
    in_universe = df[df["in_universe"] == 1].copy()
    
    groups = in_universe.groupby("timestamp")
    result = []
    
    for date, group in groups:
        factor_data = group[col_factor]
        foward_data = group[col_forward_return]
        
        for i, factor in enumerate(col_factor):
            factor_value = factor_data.loc[:,factor]
            
            for j, foward in enumerate(col_forward_return):
                foward_value = foward_data.loc[:,foward]
                
                corr, p_value = spearmanr(factor_value, foward_value)
                result.append({
                    "timestamp": date,
                    "factor_id": factor,
                    "horizon":foward,
                    "ic_spearmanr": corr  
                })
                
                
    ic_daily = pd.DataFrame(result)
    return ic_daily

daily_ic = build_daily_ic(df, col_factor, col_forward_return)
daily_ic

,timestamp,factor_id,horizon,ic_spearmanr
0,2020-09-27 00:00:00+00:00,z_momentum_7d,fwd_1d,-0.003096
1,2020-09-27 00:00:00+00:00,z_momentum_7d,fwd_5d,0.143447
2,2020-09-27 00:00:00+00:00,z_momentum_7d,fwd_14d,0.488132
3,2020-09-27 00:00:00+00:00,z_momentum_14d,fwd_1d,-0.455108
4,2020-09-27 00:00:00+00:00,z_momentum_14d,fwd_5d,-0.126935
...,...,...,...,...
16591,2021-12-31 00:00:00+00:00,z_amihud_14d,fwd_5d,0.192513
16592,2021-12-31 00:00:00+00:00,z_amihud_14d,fwd_14d,-0.008162
16593,2021-12-31 00:00:00+00:00,z_amihud_30d,fwd_1d,0.158927
16594,2021-12-31 00:00:00+00:00,z_amihud_30d,fwd_5d,0.188948


## C2  — IC Inference: Mean, Newey-West t-stat
Sau khi có spearman theo ngày của từng factor ứng với mỗi horizon, ta xây dựng thêm bảng thống kê thể hiện chỉ số ic_mean, ic_std, ic_ir:  
- ic_mean: tương ứng giá trị trung bình của coeff_spearman của 1 factor với 1 horizon trong toàn bộ khoảng thời gian  
- ic_std: tương tự ic_mean nhưng cho ic_std  
- ic_ir: ic_mean / ic_std   
- Và kiểm định thống kê ý nghĩa ic_mean  

In [5]:
""" 
Mục tiêu của hàm này là tạo ra bảng thống kê của ic_daily (IC: information coeff)
Những chỉ số quan trọng là: ic_mean, ic_std, ic_ir = ic_mean/ic_std, ic_t_stat
"""
def build_statistics_ic(daily_ic):
    df = daily_ic.copy()
    
    temp = df.groupby(["factor_id","horizon"]).agg(
        ic_mean= ("ic_spearmanr", 'mean'),
        ic_std = ("ic_spearmanr", "std")
    )
    
    temp["ic_ir"] = temp["ic_mean"] / temp["ic_std"]
    return temp


def calculate_nw_statistics(daily_ic, lag_map=None):

    if lag_map is None:
        lag_map = {
            "fwd_1d": 0,
            "fwd_5d": 4,
            "fwd_14d": 13
        }

    results = []

    for (factor_id, foward_id), group in daily_ic.groupby(
        ["factor_id", "horizon"]
    ):

        ic = (
            group["ic_spearmanr"]
            .dropna()
            .astype(float)
            .values
        )

        n_days = len(ic)

        if n_days < 2:
            results.append({
                "factor_id": factor_id,
                "horizon": foward_id,
                "nw_lag": np.nan,
                "nw_se": np.nan,
                "nw_t_stat": np.nan,
                "n_days": n_days
            })
            continue

        lag = min(
            lag_map[foward_id],
            n_days - 1
        )

        X = np.ones((n_days, 1))

        model = sm.OLS(ic, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": lag}
        )

        results.append({
            "factor_id": factor_id,
            "horizon": foward_id,
            "nw_lag": lag,
            "nw_se": model.bse[0],
            "nw_t_stat": model.tvalues[0],
            "n_days": n_days
        })

    return pd.DataFrame(results)



def add_p_value(nw_stats):

    df = nw_stats.copy()

    df["p_value"] = (
        2 * t.sf(
            np.abs(df["nw_t_stat"]),
            df["n_days"] - 1
        )
    )

    return df


def build_ic_summary(daily_ic):

    statistics = (
        build_statistics_ic(daily_ic)
        .reset_index()
    )

    nw_stats = calculate_nw_statistics(daily_ic)

    nw_stats = add_p_value(nw_stats)

    summary = (
        statistics
        .merge(
            nw_stats,
            on=["factor_id", "horizon"],
            how="left"
        )
    )
    
    def sig_star(p):
        if p < 0.01:
            return 'alpha 1%'
        elif p < 0.05:
            return 'alpha 5%'
        else:
            return 'not significant'
    summary["sig"] = summary["p_value"].apply(lambda x: sig_star(x))

    return summary



ic_statistics = build_ic_summary(daily_ic)
ic_statistics

,factor_id,horizon,ic_mean,ic_std,ic_ir,nw_lag,nw_se,nw_t_stat,n_days,p_value,sig
0,z_amihud_14d,fwd_14d,-0.128100,0.288763,-0.443617,13,0.051812,-2.472402,297,1.398279e-02,alpha 5%
1,z_amihud_14d,fwd_1d,-0.081752,0.274573,-0.297740,0,0.015906,-5.139827,297,5.003979e-07,alpha 1%
2,z_amihud_14d,fwd_5d,-0.120583,0.248932,-0.484401,4,0.025318,-4.762672,297,2.995776e-06,alpha 1%
3,z_amihud_30d,fwd_14d,-0.133038,0.328809,-0.404606,13,0.072981,-1.822928,201,6.980736e-02,not significant
4,z_amihud_30d,fwd_1d,-0.080167,0.310089,-0.258529,0,0.021818,-3.674428,201,3.060853e-04,alpha 1%
5,z_amihud_30d,fwd_5d,-0.125999,0.279668,-0.450531,4,0.035116,-3.588131,201,4.188289e-04,alpha 1%
6,z_momentum_14d,fwd_14d,0.045414,0.274009,0.165740,13,0.040104,1.132402,297,2.583820e-01,not significant
7,z_momentum_14d,fwd_1d,0.000517,0.303573,0.001703,0,0.017585,0.029406,297,9.765606e-01,not significant
8,z_momentum_14d,fwd_5d,0.030688,0.274307,0.111874,4,0.027333,1.122745,297,2.624556e-01,not significant
9,z_momentum_30d,fwd_14d,0.011552,0.259315,0.044549,13,0.044782,0.257964,201,7.966996e-01,not significant


## C3 — Univariate Fama–MacBeth

Phần này trả lời cho câu hỏi: theo C2 thì những factor có quan hệ với foward return thì ở phần C3 này sẽ trả lời nếu có thì sẽ biểu
diễn thành quan hệ tuyến tính thế nào  

- Ta xây dựng mô hình hồi quy cho từng ngày, ứng với mỗi ngày t chọn ra các coin hiện có trong UNIVERSE và tạo mô hình hồi quy dựa vào factor và target là horizon (gọi hệ số của factor là beta i)  

- Sau đó, trên toàn bộ khoảng thời gian ta tính mean của beta i  
- Đồng thời kiểm định giá trị beta đó có ý nghĩa thống kê hay không  


In [ ]:
# Ta sẽ thực hiện winsorize cho các forward return để loại bỏ các giá trị outlier ảnh hưởng tới phần xây dựng model hồi quy về sau
def winsorize_series(series, lower= 0.05, upper= 0.95):
    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)
    return series.clip(lower = lower_bound, upper = upper_bound)

def winsorize_fwd(df, col_forward_return):
    temp = df.copy()
    for fwd in col_forward_return:
        temp[fwd] = temp.groupby("timestamp")[fwd].transform(winsorize_series)
    return temp

df_regression = winsorize_fwd(df, col_forward_return)
df_regression # dataset phục vụ riêng cho phần regression phía sau

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d,fwd_1d,fwd_5d,fwd_14d
4851,2020-09-27 00:00:00+00:00,BTC/USDT,1,0.104937,1.295379,0.608275,-0.643410,0.059951,-0.801625,-1.335920,-1.460214,-1.898988,-1.592089,0.004246,-1.130024,-1.219553,-0.007278,-0.019101,0.053733
4852,2020-09-27 00:00:00+00:00,ETH/USDT,1,-0.304986,0.608550,0.439487,0.332203,0.293393,-0.199482,-0.491809,-0.556091,-0.670204,-1.220259,0.010036,-1.124777,-1.213106,-0.009923,-0.033825,0.045576
4853,2020-09-27 00:00:00+00:00,LINK/USDT,1,2.119062,-0.288078,-0.790309,1.667830,1.710906,1.711488,1.784511,1.734227,1.347208,1.230560,0.045198,-0.919622,-1.009862,-0.048489,-0.131439,0.005322
4854,2020-09-27 00:00:00+00:00,BNB/USDT,1,0.294696,-0.908935,1.626324,0.596841,-0.032517,1.024796,0.163254,0.335636,0.197070,-1.114051,0.001952,-0.950499,-0.984128,0.027596,0.040533,0.083150
4855,2020-09-27 00:00:00+00:00,TRX/USDT,1,0.111595,-0.604139,1.507666,0.486506,-0.995973,-0.528470,-1.229117,-0.538236,0.136846,0.244343,-0.021946,-0.706681,-0.747670,-0.012869,-0.026292,-0.001129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48483,2021-12-31 00:00:00+00:00,ACM/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011087,NaN,NaN,0.036744,-0.060737,-0.031550
48484,2021-12-31 00:00:00+00:00,RIF/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011526,NaN,NaN,0.038076,-0.079707,-0.098440
48485,2021-12-31 00:00:00+00:00,ONG/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011306,NaN,NaN,0.049198,-0.001874,-0.044586
48486,2021-12-31 00:00:00+00:00,ASR/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.007439,NaN,NaN,0.049288,-0.053680,-0.044011


In [19]:

""" 
Ta thực hiện chạy hồi quy cho toàn bộ factor cho từng ngày, ứng với mỗi factor ta kiểm tra độ tuyến tính với từng foward return và mối quan hệ đó có 
ý nghĩa thống kê hay không.
Chỉ thực hiện trên nhãn in_universe == 1
Ví dụ ngày t, ta muốn đo độ tuyến tính giũa momentum_7d với fwd_5d và cần biết có ý nghĩa thống kê hay không.
"""

def build_regression_daily(df, col_factor, col_forward_return):
    in_universe = df[df["in_universe"] == 1].copy()
    
    results = []
    for factor in col_factor:
        for forward in col_forward_return:
            beta_series= []
            for date, group in in_universe.groupby("timestamp"):
                group_regression = group.dropna(subset=[factor, forward])
                x = sm.add_constant(group_regression[factor])
                y = group_regression[forward]
                
                model = sm.OLS(y, x).fit()
                beta_t = model.params.iloc[1]
                alpha_t = model.params.iloc[0]
                
                beta_series.append({
                    "date": date,
                    "beta_t": beta_t,
                    "alpha_t": alpha_t,
                    "n_obs": len(group_regression)
                })                    
            beta_df = pd.DataFrame(beta_series)
            
            
            
            n_days = len(beta_df)
            beta_mean = beta_df['beta_t'].mean()
            beta_std = beta_df['beta_t'].std()
            alpha_mean = beta_df['alpha_t'].mean()
            
            # NEWEY-WEST T-STAT
            # Xác định số lag dựa trên forward horizon
            if forward == 'fwd_1d':
                maxlag = 0
            elif forward == 'fwd_5d':
                maxlag = 4
            elif forward == 'fwd_14d':
                maxlag = 13
            else:
                maxlag = 0
            
            y_nw = beta_df['beta_t'].values
            X_nw = np.ones((n_days, 1))
            
            try:
                model_nw = sm.OLS(y_nw, X_nw).fit(
                    cov_type='HAC',
                    cov_kwds={'maxlags': maxlag}
                )
                nw_se = model_nw.bse[0]
                nw_t_stat = model_nw.tvalues[0]
            except:
                nw_se = beta_std / np.sqrt(n_days)
                nw_t_stat = beta_mean / nw_se
            
            # p-value (2-tailed)
            p_value = 2 * t.sf(abs(nw_t_stat), n_days - 1)
            
            results.append({
                'factor_id': factor,
                'horizon': forward,
                'beta_mean': beta_mean,
                'beta_std': beta_std,
                'nw_se': nw_se,
                'nw_t_stat': nw_t_stat,
                'p_value': p_value,
                'alpha_mean': alpha_mean,
                'n_days': n_days,
                'avg_n_obs': beta_df['n_obs'].mean()
            })
    
    fm_summary = pd.DataFrame(results)
    
    def sig_star(p):
        if p < 0.01:
            return 'alpha 1%'
        elif p < 0.05:
            return 'alpha 5%'
        else:
            return 'not significant'
    
    fm_summary['sig'] = fm_summary['p_value'].apply(sig_star)
    
    # Sắp xếp
    horizon_order = {'fwd_1d': 0, 'fwd_5d': 1, 'fwd_14d': 2}
    fm_summary['horizon_rank'] = fm_summary['horizon'].map(horizon_order)
    fm_summary = fm_summary.sort_values(['factor_id', 'horizon_rank'])
    fm_summary = fm_summary.drop(columns=['horizon_rank'])
    
    return fm_summary.reset_index(drop=True)

build_regression_daily(df_regression, col_factor, col_forward_return)

,factor_id,horizon,beta_mean,beta_std,nw_se,nw_t_stat,p_value,alpha_mean,n_days,avg_n_obs,sig
0,z_amihud_14d,fwd_1d,-0.000475,0.009797,0.000456,-1.042654,0.297656,0.001074,461,33.110629,not significant
1,z_amihud_14d,fwd_5d,-0.001516,0.022356,0.001914,-0.792254,0.428621,0.010200,461,33.110629,not significant
2,z_amihud_14d,fwd_14d,-0.003879,0.038467,0.005429,-0.714492,0.475285,0.030230,461,33.110629,not significant
3,z_amihud_30d,fwd_1d,-0.000532,0.010664,0.000496,-1.072865,0.283894,0.001088,461,32.409978,not significant
4,z_amihud_30d,fwd_5d,-0.001142,0.025836,0.002277,-0.501463,0.616285,0.010332,461,32.409978,not significant
5,z_amihud_30d,fwd_14d,-0.004319,0.044691,0.006239,-0.692260,0.489123,0.030849,461,32.409978,not significant
6,z_momentum_14d,fwd_1d,0.000437,0.012435,0.000579,0.755350,0.450426,0.001074,461,33.110629,not significant
7,z_momentum_14d,fwd_5d,0.002670,0.028082,0.002380,1.121725,0.262565,0.010200,461,33.110629,not significant
8,z_momentum_14d,fwd_14d,0.006295,0.043882,0.005289,1.190199,0.234582,0.030230,461,33.110629,not significant
9,z_momentum_30d,fwd_1d,-0.000066,0.012432,0.000578,-0.113545,0.909648,0.001088,461,32.409978,not significant


## C4 — Quintile Portfolio Spread


Vì ta sẽ thực hiện mô hình tuyến tính ở các phần sau nên việc xác định dấu mong muốn của 1 factor với 1 horizon là cần thiết  

Ta kỳ vọng dấu giữa spread_mean, beta_mean, ic_mean cần chung dấu (vì ta mong muốn diễn giải được quan hệ tuyến tính theo kỳ vọng):  
- Nghĩa là khi spread_mean âm giữa một factor x một horizon, đồng nghĩa z_score của factor đó càng lớn thì horizon đó càng nhỏ  
- Nếu spread_mean dương thì z_socre của factor đó càng lớn thì horizon càng lớn  
- beta_mean dương nghĩa là xu hướng giữa factor và horizon, factor càng lớn thì horizon cũng có xu hướng tăng theo  
     
Đồng thời thêm điều kiện chỉ chọn factor x horizon mà có spread_mean đủ độ lớn để bù phần chi phí khác (ví dụ chi phí giao dịch)  


In [20]:
""" 
Ứng với ngày t, của factor i, ta chia giá trị của factor i đang xét thành 3 nhóm High, Low,  Mid
Sắp xếp theo chiều giá trị factor tăng dần, và gán xem coin nào thuộc nhóm nào
Spread = average_foward_return (nhóm Hight) - average_forward_return (nhóm Low) 

Sau đó tổng hợp qua toàn bộ thời gian, tính spread_mean, spread_Std và ý nghĩa thống kê
"""
def build_day_portfolio_spread(df, col_factor, col_forward_return, n_group = 3):
    in_universe = df[df["in_universe"] == 1].copy()

    results = []
    for date, group in in_universe.groupby("timestamp"):
        group = group.dropna()
        n_coins = len(group)
        for factor in col_factor:
            
            factor_group = group.sort_values(factor)
            
            factor_group["group"] = pd.qcut(np.arange(n_coins), q = n_group, labels= range(n_group))
            
            for forward_return in col_forward_return:
                group_mean = factor_group.groupby("group")[forward_return].mean()
                
                avg_low = group_mean.iloc[0]
                avg_mid = group_mean.iloc[1]
                avg_high = group_mean.iloc[2]
                
                    
                spread = avg_high - avg_low
                
                results.append({
                    'date': date,
                    'factor_id': factor,
                    'forward_return': forward_return,
                    'spread': spread,
                    'avg_return_low': avg_low,
                    'avg_return_mid': avg_mid,
                    'avg_return_high': avg_high
                })
    return pd.DataFrame(results)

day_portfolio_spread = build_day_portfolio_spread(df, col_factor, col_forward_return)
day_portfolio_spread

,date,factor_id,forward_return,spread,avg_return_low,avg_return_mid,avg_return_high
0,2020-09-27 00:00:00+00:00,z_momentum_7d,fwd_1d,0.002846,-0.008711,0.001441,-0.005866
1,2020-09-27 00:00:00+00:00,z_momentum_7d,fwd_5d,0.035427,-0.077106,-0.007482,-0.041679
2,2020-09-27 00:00:00+00:00,z_momentum_7d,fwd_14d,0.135136,-0.055395,0.085067,0.079741
3,2020-09-27 00:00:00+00:00,z_momentum_14d,fwd_1d,-0.020552,0.005435,-0.003454,-0.015117
4,2020-09-27 00:00:00+00:00,z_momentum_14d,fwd_5d,-0.025400,-0.044074,-0.012719,-0.069474
...,...,...,...,...,...,...,...
16591,2021-12-31 00:00:00+00:00,z_amihud_14d,fwd_5d,0.045462,-0.073665,-0.019891,-0.028203
16592,2021-12-31 00:00:00+00:00,z_amihud_14d,fwd_14d,0.045132,-0.071860,-0.009433,-0.026729
16593,2021-12-31 00:00:00+00:00,z_amihud_30d,fwd_1d,0.018076,0.037881,0.045127,0.055957
16594,2021-12-31 00:00:00+00:00,z_amihud_30d,fwd_5d,0.036967,-0.073665,-0.011396,-0.036698


In [21]:
def build_portfolio(day_portfolio_spread):
    
    horizon_lag_map = {
            'fwd_1d': 0,
            'fwd_5d': 4,
            'fwd_14d': 13
        }
    
    df = day_portfolio_spread.copy()
    
    results = []
    
    for (factor_id, horizon), group in df.groupby(['factor_id', 'forward_return']):
        group = group.sort_values('date')
        spread_series = group['spread'].dropna()
        n_days = len(spread_series)
        
        spread_mean = spread_series.mean()
        spread_std = spread_series.std()
        
        lag = min(horizon_lag_map.get(horizon, 0), n_days - 1)
        x = np.ones((n_days, 1))
        
        model = sm.OLS(spread_series.values, x).fit(
                cov_type='HAC',
                cov_kwds={'maxlags': lag}
            )
        nw_se = model.bse[0]
        nw_t_stat = model.tvalues[0]
        
        p_value = 2 * t.sf(abs(nw_t_stat), n_days - 1)
        
        
        results.append({
            'factor_id': factor_id,
            'horizon': horizon,
            'spread_mean': spread_mean,
            'spread_std': spread_std,
            'nw_se': nw_se,
            'nw_t_stat': nw_t_stat,
            'p_value': p_value
        })
    
    summary = pd.DataFrame(results)
    
    def sig_star(p):
        if p < 0.01:
            return "alpha 1%"
        elif p < 0.05:
            return "alpha 5%"
        else:
            return "not significant"
    
    summary['sig'] = summary['p_value'].apply(sig_star)
    

    horizon_order = {'fwd_1d': 0, 'fwd_5d': 1, 'fwd_14d': 2}
    summary['horizon_rank'] = summary['horizon'].map(horizon_order)
    summary = summary.sort_values(['factor_id', 'horizon_rank'])
    summary = summary.drop(columns=['horizon_rank'])
    
    return summary.reset_index(drop=True)

build_portfolio(day_portfolio_spread)

,factor_id,horizon,spread_mean,spread_std,nw_se,nw_t_stat,p_value,sig
0,z_amihud_14d,fwd_1d,0.000238,0.029324,0.001364,0.174128,0.861842,not significant
1,z_amihud_14d,fwd_5d,0.002035,0.065643,0.005661,0.359541,0.719355,not significant
2,z_amihud_14d,fwd_14d,0.005410,0.114430,0.016553,0.326840,0.743938,not significant
3,z_amihud_30d,fwd_1d,0.000332,0.030791,0.001433,0.231891,0.816726,not significant
4,z_amihud_30d,fwd_5d,0.001907,0.069120,0.006066,0.314384,0.753372,not significant
5,z_amihud_30d,fwd_14d,0.002360,0.128687,0.018342,0.128670,0.897675,not significant
6,z_momentum_14d,fwd_1d,0.002437,0.033978,0.001581,1.541460,0.123892,not significant
7,z_momentum_14d,fwd_5d,0.010676,0.075003,0.006382,1.672768,0.095053,not significant
8,z_momentum_14d,fwd_14d,0.022195,0.118866,0.014052,1.579512,0.114906,not significant
9,z_momentum_30d,fwd_1d,0.002587,0.034234,0.001593,1.624226,0.105013,not significant


In [27]:
def build_summary_factor(df, col_factor, col_forward_return):
    
    df_regression = winsorize_fwd(df, col_forward_return)
    daily_ic = build_daily_ic(df, col_factor, col_forward_return)
    ic_summary = (build_ic_summary(daily_ic))[["factor_id","horizon","ic_mean","sig"]]
    regression_summary = (build_regression_daily(df_regression, col_factor, col_forward_return))[["factor_id","horizon", "beta_mean","sig"]]
    day_portfolio_spread = build_day_portfolio_spread(df, col_factor, col_forward_return)
    portfolio_summary = (build_portfolio(day_portfolio_spread))[["factor_id","horizon","spread_mean","sig"]]
    
    result = (portfolio_summary.merge(ic_summary, on = ["factor_id","horizon"], how ="inner")).merge(
        regression_summary, on = ["factor_id","horizon"], how ="inner")
    
    return result

summary_factor = build_summary_factor(df, col_factor, col_forward_return)
summary_factor

,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig
0,z_amihud_14d,fwd_1d,0.000238,not significant,-0.081752,alpha 1%,-0.000475,not significant
1,z_amihud_14d,fwd_5d,0.002035,not significant,-0.120583,alpha 1%,-0.001516,not significant
2,z_amihud_14d,fwd_14d,0.005410,not significant,-0.128100,alpha 5%,-0.003879,not significant
3,z_amihud_30d,fwd_1d,0.000332,not significant,-0.080167,alpha 1%,-0.000532,not significant
4,z_amihud_30d,fwd_5d,0.001907,not significant,-0.125999,alpha 1%,-0.001142,not significant
5,z_amihud_30d,fwd_14d,0.002360,not significant,-0.133038,not significant,-0.004319,not significant
6,z_momentum_14d,fwd_1d,0.002437,not significant,0.000517,not significant,0.000437,not significant
7,z_momentum_14d,fwd_5d,0.010676,not significant,0.030688,not significant,0.002670,not significant
8,z_momentum_14d,fwd_14d,0.022195,not significant,0.045414,not significant,0.006295,not significant
9,z_momentum_30d,fwd_1d,0.002587,not significant,0.018743,not significant,-0.000066,not significant


In [28]:
def cal_speard_mean_day(row):
    if row["horizon"] == "fwd_1d":
        return 1
    elif row["horizon"] == "fwd_5d":
        return 5
    else:
        return 14

summary_factor_1 = summary_factor.copy()
summary_factor_1["spread_mean_magnitude"] = summary_factor_1.apply(lambda x: abs(x["spread_mean"] *10000)/ cal_speard_mean_day(x) , axis = 1)
summary_factor_1



,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig,spread_mean_magnitude
0,z_amihud_14d,fwd_1d,0.000238,not significant,-0.081752,alpha 1%,-0.000475,not significant,2.375593
1,z_amihud_14d,fwd_5d,0.002035,not significant,-0.120583,alpha 1%,-0.001516,not significant,4.070713
2,z_amihud_14d,fwd_14d,0.005410,not significant,-0.128100,alpha 5%,-0.003879,not significant,3.864349
3,z_amihud_30d,fwd_1d,0.000332,not significant,-0.080167,alpha 1%,-0.000532,not significant,3.321947
4,z_amihud_30d,fwd_5d,0.001907,not significant,-0.125999,alpha 1%,-0.001142,not significant,3.814147
5,z_amihud_30d,fwd_14d,0.002360,not significant,-0.133038,not significant,-0.004319,not significant,1.685774
6,z_momentum_14d,fwd_1d,0.002437,not significant,0.000517,not significant,0.000437,not significant,24.367254
7,z_momentum_14d,fwd_5d,0.010676,not significant,0.030688,not significant,0.002670,not significant,21.352178
8,z_momentum_14d,fwd_14d,0.022195,not significant,0.045414,not significant,0.006295,not significant,15.853430
9,z_momentum_30d,fwd_1d,0.002587,not significant,0.018743,not significant,-0.000066,not significant,25.869412


Thực hiện chọn factor theo hướng:
- Có ít nhất 1 kiểm định ý nghĩa alpha 5%  
- spread_mean, beta, ic_mean cần cùng dấu -> bắt quan hệ tuyến tính cho mô hình hồi quy phía sau  
- độ lớn của spread_mean đủ lớn để bù các chi phí giao dịch...  

In [53]:
def check_sign(x):
    if x > 0:
        return 1
    elif x< 0:
        return -1
    else: 
        return 0

# input là mỗi dòng trong bảng summary_factor
def check_quality_factor(row):
    sig_x = (row["sig_x"] == "alpha 1%") or (row["sig_x"] == "alpha 5%")
    sig_y = (row["sig_y"] == "alpha 1%") or (row["sig_y"] == "alpha 5%")
    sig = row["sig"] == ("alpha 1%") or (row["sig"] == "alpha 5%")
    n_sig = sum([sig_x, sig_y, sig]) # số kiểm định hợp lệ
    
    spread_mean_sign = check_sign(row["spread_mean"])
    ic_mean_sign = check_sign(row["ic_mean"])
    beta_mean_sign = check_sign(row["beta_mean"])
    
    sign = [spread_mean_sign, ic_mean_sign, beta_mean_sign]
    sign_consistence = 1 if (sum(sign) == 3 or sum(sign) == -3) else 0 # cùng chiều hay k
    
    is_spread_enough = row["spread_mean_magnitude"] >= 10
    
    if n_sig >= 1 and sign_consistence == 1 and is_spread_enough:
        return True
    else:
        return False

summary_factor_1["eligible"] = summary_factor_1.apply(lambda x: check_quality_factor(x), axis = 1)
summary_factor_1["sign_ortention"] = summary_factor_1.apply(lambda x: 1 if (check_sign(x["spread_mean"]) + check_sign(x["ic_mean"]) 
                                                                                      + check_sign(x["beta_mean"])) > 0 else -1 , axis = 1)

summary_factor_1

,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig,spread_mean_magnitude,eligible,sign_ortention
0,z_amihud_14d,fwd_1d,0.000238,not significant,-0.081752,alpha 1%,-0.000475,not significant,2.375593,False,-1
1,z_amihud_14d,fwd_5d,0.002035,not significant,-0.120583,alpha 1%,-0.001516,not significant,4.070713,False,-1
2,z_amihud_14d,fwd_14d,0.005410,not significant,-0.128100,alpha 5%,-0.003879,not significant,3.864349,False,-1
3,z_amihud_30d,fwd_1d,0.000332,not significant,-0.080167,alpha 1%,-0.000532,not significant,3.321947,False,-1
4,z_amihud_30d,fwd_5d,0.001907,not significant,-0.125999,alpha 1%,-0.001142,not significant,3.814147,False,-1
5,z_amihud_30d,fwd_14d,0.002360,not significant,-0.133038,not significant,-0.004319,not significant,1.685774,False,-1
6,z_momentum_14d,fwd_1d,0.002437,not significant,0.000517,not significant,0.000437,not significant,24.367254,False,1
7,z_momentum_14d,fwd_5d,0.010676,not significant,0.030688,not significant,0.002670,not significant,21.352178,False,1
8,z_momentum_14d,fwd_14d,0.022195,not significant,0.045414,not significant,0.006295,not significant,15.853430,False,1
9,z_momentum_30d,fwd_1d,0.002587,not significant,0.018743,not significant,-0.000066,not significant,25.869412,False,1


In [54]:
factor_eligible = summary_factor_1[summary_factor_1["eligible"] == True].reset_index(drop= True)
factor_eligible

,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig,spread_mean_magnitude,eligible,sign_ortention
0,z_reversal_1d,fwd_14d,0.015871,alpha 5%,0.023505,not significant,0.002926,not significant,11.336563,True,1
1,z_reversal_3d,fwd_5d,0.012721,alpha 5%,0.018627,not significant,0.003692,alpha 5%,25.441549,True,1
2,z_reversal_3d,fwd_14d,0.019971,alpha 5%,0.026023,not significant,0.005135,not significant,14.264880,True,1
3,z_vol_14d,fwd_1d,-0.001126,not significant,-0.088336,alpha 1%,-0.001225,alpha 5%,11.256978,True,-1
4,z_vol_14d,fwd_5d,-0.005671,not significant,-0.104530,alpha 1%,-0.003442,not significant,11.342516,True,-1
5,z_vol_14d,fwd_14d,-0.022476,not significant,-0.100844,alpha 5%,-0.009209,not significant,16.054214,True,-1
6,z_vol_30d,fwd_1d,-0.001725,not significant,-0.071607,alpha 1%,-0.001612,alpha 1%,17.251893,True,-1
7,z_vol_30d,fwd_5d,-0.008122,not significant,-0.079176,alpha 5%,-0.005240,alpha 5%,16.244326,True,-1
8,z_vol_30d,fwd_14d,-0.031765,not significant,-0.084052,not significant,-0.012820,alpha 5%,22.689496,True,-1
9,z_vol_7d,fwd_5d,-0.006088,not significant,-0.089626,alpha 1%,-0.003974,alpha 5%,12.175294,True,-1


In [56]:
# chỉ lọc ra những factor đủ điều kiện
col_factor_qualified = factor_eligible["factor_id"].unique().tolist()


new_df = df[["timestamp","symbol","in_universe"]+ col_factor_qualified + col_forward_return].copy()
new_df

,timestamp,symbol,in_universe,z_reversal_1d,z_reversal_3d,z_vol_14d,z_vol_30d,z_vol_7d,z_vol_of_vol_14d,fwd_1d,fwd_5d,fwd_14d
4851,2020-09-27 00:00:00+00:00,BTC/USDT,1,0.059951,-0.801625,-1.460214,-1.898988,-1.335920,-1.592089,-0.007278,-0.019101,0.053733
4852,2020-09-27 00:00:00+00:00,ETH/USDT,1,0.293393,-0.199482,-0.556091,-0.670204,-0.491809,-1.220259,-0.009923,-0.033825,0.045576
4853,2020-09-27 00:00:00+00:00,LINK/USDT,1,1.710906,1.711488,1.734227,1.347208,1.784511,1.230560,-0.048489,-0.157964,0.005322
4854,2020-09-27 00:00:00+00:00,BNB/USDT,1,-0.032517,1.024796,0.335636,0.197070,0.163254,-1.114051,0.027596,0.040533,0.083150
4855,2020-09-27 00:00:00+00:00,TRX/USDT,1,-0.995973,-0.528470,-0.538236,0.136846,-1.229117,0.244343,-0.012869,-0.026292,-0.001129
...,...,...,...,...,...,...,...,...,...,...,...,...
48483,2021-12-31 00:00:00+00:00,ACM/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.036744,-0.060737,-0.031550
48484,2021-12-31 00:00:00+00:00,RIF/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.038076,-0.079707,-0.098440
48485,2021-12-31 00:00:00+00:00,ONG/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.049198,-0.001874,-0.044586
48486,2021-12-31 00:00:00+00:00,ASR/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.049288,-0.053680,-0.044011


In [57]:
"""Thực hiện kiểm định corr cho toàn bộ những biến vừa được chọn"""
factor_eligible_1day = factor_eligible[factor_eligible["horizon"] == "fwd_1d"]


def check_corr(df, col_factor_qualified):
    in_universe = df[df["in_universe"] == 1].copy()

    daily_corr = (
        in_universe
        .groupby("timestamp")[col_factor_qualified]
        .corr(method="spearman")
        .rename_axis(["timestamp", "factor"])
    )

    median_corr = daily_corr.groupby(level="factor").median()

    upper_corr = median_corr.where(
        np.triu(np.ones(median_corr.shape), k=1).astype(bool)
    )
    
    # Chuyển matrix sang dạng factor_a, factor_b, corr
    corr_pair = (
        upper_corr
        .stack()
        .reset_index()
    )

    corr_pair.columns = ["factor_a", "factor_b", "corr"]


    return corr_pair.dropna()
corr_factor = check_corr(new_df, col_factor_qualified)
corr_factor
new = corr_factor.merge(factor_eligible_1day[["factor_id","spread_mean","ic_mean"]], left_on="factor_a", right_on="factor_id", how = 'inner' )
new = new.merge(factor_eligible_1day[["factor_id","spread_mean","ic_mean"]], left_on="factor_b", right_on="factor_id", how = 'inner')

new = new.rename(columns={"spread_mean_x": "spread_mean_a",
                     "ic_mean_x": "ic_mean_a","spread_mean_y":"spread_mean_b","ic_mean_y":"ic_mean_b" })

new.drop(columns=["factor_id_x","factor_id_y"])

,factor_a,factor_b,corr,spread_mean_a,ic_mean_a,spread_mean_b,ic_mean_b
0,z_vol_14d,z_vol_30d,0.814646,-0.001126,-0.088336,-0.001725,-0.071607
1,z_vol_14d,z_vol_of_vol_14d,0.530571,-0.001126,-0.088336,-0.001931,-0.046737
2,z_vol_30d,z_vol_of_vol_14d,0.632724,-0.001725,-0.071607,-0.001931,-0.046737


- Bỏ đi z_vol_30d (vì ic_mean của z_vol_14d lớn hơn)  


In [52]:
col_factor_qualified = ["z_vol_14d","z_vol_of_vol_14d"]

new_df= df[["timestamp","symbol","in_universe"] + col_factor_qualified + col_forward_return]
new_df

,timestamp,symbol,in_universe,z_vol_14d,z_vol_of_vol_14d,fwd_1d,fwd_5d,fwd_14d
4851,2020-09-27 00:00:00+00:00,BTC/USDT,1,-1.460214,-1.592089,-0.007278,-0.019101,0.053733
4852,2020-09-27 00:00:00+00:00,ETH/USDT,1,-0.556091,-1.220259,-0.009923,-0.033825,0.045576
4853,2020-09-27 00:00:00+00:00,LINK/USDT,1,1.734227,1.230560,-0.048489,-0.157964,0.005322
4854,2020-09-27 00:00:00+00:00,BNB/USDT,1,0.335636,-1.114051,0.027596,0.040533,0.083150
4855,2020-09-27 00:00:00+00:00,TRX/USDT,1,-0.538236,0.244343,-0.012869,-0.026292,-0.001129
...,...,...,...,...,...,...,...,...
48483,2021-12-31 00:00:00+00:00,ACM/USDT,0,NaN,NaN,0.036744,-0.060737,-0.031550
48484,2021-12-31 00:00:00+00:00,RIF/USDT,0,NaN,NaN,0.038076,-0.079707,-0.098440
48485,2021-12-31 00:00:00+00:00,ONG/USDT,0,NaN,NaN,0.049198,-0.001874,-0.044586
48486,2021-12-31 00:00:00+00:00,ASR/USDT,0,NaN,NaN,0.049288,-0.053680,-0.044011


Tổng kết:
- Tất cả phần công việc phía trên cho mục đích chọn ra những Factor đủ điều kiện thống kê để từ đó việc xây dựng về sau sẽ chỉ dùng các Factor đó  
  
- Những kết quả thống kê được tập hợp từ việc:   
    - ic: information coeff có ý nghĩa thống kê hay không
    - beta: hệ số hồi quy có đủ điều kiện
    - spread: độ chênh lệch giữa nhóm z_factor_i cao và nhóm z_factor_i thấp
  

---> Sau cùng ta có các Factor đủ điều kiện cho phần sau: z_vol_14d, z_vol_of_vol_14d"